# Empire Play - Importador v3

Roda 100% no navegador via Google Colab.

1. Execute celula por celula (Shift+Enter)
2. Na celula 2, autorize com sua conta Google
3. Cole a SUPABASE_KEY quando solicitado na celula 3

In [ ]:
!pip install -q gspread supabase

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('OK autenticado')

In [ ]:
from getpass import getpass
SUPABASE_URL = 'https://rqwprnvlrobabfotmmnf.supabase.co'
SUPABASE_KEY = getpass('Supabase service_role key: ')
SHEET_ID = '1XYa6Pzd-lou3fzqaZgjhBYNb3Je2PB9Slu7ozzOghUo'
print('OK configurado')

In [ ]:
import gspread
from google.auth import default
from supabase import create_client
creds, _ = default(scopes=['https://www.googleapis.com/auth/spreadsheets.readonly','https://www.googleapis.com/auth/drive.readonly'])
gc = gspread.authorize(creds)
sh = gc.open_by_key(SHEET_ID)
sb = create_client(SUPABASE_URL, SUPABASE_KEY)
print('OK conectado')

In [ ]:
def ler(nome):
    try:
        ws = sh.worksheet(nome)
        rows = ws.get_all_records(default_blank=None)
        print(f'  OK {nome}: {len(rows)} linhas')
        return rows
    except gspread.WorksheetNotFound:
        print(f'  ABA NAO ENCONTRADA: {nome}')
        return []

def upsert(tabela, reg):
    if not reg:
        print(f'  SEM DADOS: {tabela}'); return
    for i in range(0, len(reg), 500):
        try:
            sb.table(tabela).upsert(reg[i:i+500]).execute()
            print(f'  OK {tabela} lote {i//500+1} ({len(reg[i:i+500])} registros)')
        except Exception as e:
            print(f'  ERRO {tabela}: {e}')

def debug_colunas(rows):
    if rows:
        print(f'  Colunas: {list(rows[0].keys())[:10]}')
        for k,v in list(rows[0].items())[:5]:
            print(f'    [{k}] = {repr(v)}')

def n(v):
    try: return int(v) if v else None
    except: return None

def s(v):
    return str(v).strip() if v else None

print('OK funcoes prontas')

In [ ]:
print('--- MUSICAS')
rows = ler('Musicas')
debug_colunas(rows)
reg = []
for r in rows:
    nome = (r.get('Nome da musica') or r.get('Nome da música') or
            r.get('Nome') or r.get('NOME') or r.get('Título') or r.get('Titulo'))
    if not nome: continue
    reg.append({
        'Nome': nome,
        'Artista': r.get('ACT PRINCIPAL') or r.get('Artista') or None,
        'Album': r.get('ALBUM') or r.get('Album') or None,
        'Capa da Musica': r.get('Capa da musica') or r.get('Capa da música') or None,
        'telegram_file_id': s(r.get('ID do arquivo')),
        'telegram_topic_id': n(r.get('ID do topico') or r.get('ID do tópico')),
        'genero': r.get('GENERO') or r.get('Gênero') or None,
        'tipo_single': r.get('TIPO DE SINGLE') or None,
        'tipo_musica': r.get('TIPO DE MUSICA') or r.get('TIPO DE MÚsSICA') or None,
        'data_lancamento': s(r.get('Data de lancamento') or r.get('Data de lançamento')),
        'ordem': n(r.get('Ordem')),
    })
print(f'  Registros validos: {len(reg)}')
upsert('Musicas', reg)

In [ ]:
print('--- ALBUNS')
rows = ler('Albuns')
debug_colunas(rows)
reg = []
for r in rows:
    nome = r.get('Nome')
    if not nome: continue
    reg.append({
        'Nome do Album': nome,
        'Nome do Artista': r.get('Nome do criador') or None,
        'Capa do Album': r.get('Capa') or None,
        'telegram_topic_id': n(r.get('ID do topico') or r.get('ID do tópico')),
        'data_lancamento': s(r.get('Data de lancamento') or r.get('Data de lançamento')),
    })
print(f'  Registros validos: {len(reg)}')
upsert('Albuns', reg)

In [ ]:
print('--- MUSIC VIDEOS')
rows = ler('Music Videos')
debug_colunas(rows)
reg = []
for r in rows:
    titulo = r.get('Nome') or r.get('Titulo') or r.get('Título')
    if not titulo: continue
    reg.append({
        'Titulo': titulo,
        'Artista': r.get('Nome do criador') or None,
        'Capa': r.get('Thumb') or None,
        'telegram_file_id': s(r.get('ID do arquivo')),
        'telegram_topic_id': n(r.get('ID do topico') or r.get('ID do tópico')),
        'tipo': r.get('Tipo') or None,
        'data_lancamento': s(r.get('Data de lancamento') or r.get('Data de lançamento')),
    })
print(f'  Registros validos: {len(reg)}')
upsert('Music Videos', reg)

In [ ]:
print('--- VIDEOS')
rows = ler('Videos')
debug_colunas(rows)
reg = []
for r in rows:
    titulo = r.get('titulo') or r.get('Titulo') or r.get('Nome')
    if not titulo: continue
    reg.append({
        'Titulo': titulo,
        'Artista': r.get('artista') or r.get('Artista') or r.get('enviado_por') or None,
        'Capa': r.get('thumbnail_url') or r.get('Thumb') or r.get('Capa') or None,
        'telegram_file_id': s(r.get('ID do arquivo') or r.get('telegram_file_id')),
        'telegram_topic_id': n(r.get('ID do topico') or r.get('telegram_topic_id')),
    })
print(f'  Registros validos: {len(reg)}')
upsert('Videos', reg)

In [ ]:
print('--- TOP 50 SPOTIFY')
rows = ler('Top_50_Spotify')
debug_colunas(rows)
reg = []
for i,r in enumerate(rows):
    nome = (r.get('Nome da musica') or r.get('Nome da música') or
            r.get('nome_musica') or r.get('Nome'))
    if not nome: continue
    reg.append({
        'posicao': n(r.get('Posicao') or r.get('Posição') or r.get('posicao')) or i+1,
        'nome_musica': nome,
        'capa_musica': r.get('Capa da musica') or r.get('Capa da música') or r.get('capa_musica') or None,
        'link_audio': r.get('Link do audio') or r.get('link_audio') or None,
        'telegram_topic_id': n(r.get('ID do topico') or r.get('telegram_topic_id')),
    })
print(f'  Registros validos: {len(reg)}')
upsert('Top_50_Spotify', reg)

In [ ]:
print('--- TOP APPLE MUSIC')
rows = ler('Top_Songs_Apple_Music')
debug_colunas(rows)
reg = []
for i,r in enumerate(rows):
    nome = (r.get('Nome da musica') or r.get('Nome da música') or
            r.get('nome_musica') or r.get('Nome'))
    if not nome: continue
    reg.append({
        'posicao': n(r.get('Posicao') or r.get('Posição') or r.get('posicao')) or i+1,
        'nome_musica': nome,
        'capa_musica': r.get('Capa da musica') or r.get('Capa da música') or r.get('capa_musica') or None,
        'link_audio': r.get('Link do audio') or r.get('link_audio') or None,
        'telegram_topic_id': n(r.get('ID do topico') or r.get('telegram_topic_id')),
    })
print(f'  Registros validos: {len(reg)}')
upsert('Top_Songs_Apple_Music', reg)

In [ ]:
print('--- TOP VIDEOS YT')
rows = ler('Top_Videos_YT')
debug_colunas(rows)
reg = []
for i,r in enumerate(rows):
    nome = r.get('Nome do video') or r.get('Nome do vídeo') or r.get('nome_video') or r.get('Nome')
    if not nome: continue
    reg.append({
        'posicao': n(r.get('Posicao') or r.get('Posição') or r.get('posicao')) or i+1,
        'nome_video': nome,
        'thumb': r.get('Thumb') or r.get('thumb') or None,
        'link_audio': r.get('Link do audio') or r.get('link_audio') or None,
        'telegram_topic_id': n(r.get('ID do topico') or r.get('telegram_topic_id')),
    })
print(f'  Registros validos: {len(reg)}')
upsert('Top_Videos_YT', reg)

In [ ]:
print('--- COMENTARIOS MUSICAS')
rows = ler('Comentarios_Musicas')
debug_colunas(rows)
reg = []
for r in rows:
    c = None
    for k in r:
        if k.lower().startswith('coment'): c = r[k]; break
    if not c: continue
    reg.append({
        'telegram_topic_id': n(r.get('ID do topico') or r.get('ID do tópico')),
        'id_jogador': s(r.get('ID do jogador')),
        'nome_jogador': r.get('Nome do jogador') or None,
        'comentario': c,
    })
print(f'  Registros validos: {len(reg)}')
upsert('Comentarios_Musicas', reg)

In [ ]:
print('--- COMENTARIOS MV')
rows = ler('Comentarios_MV')
debug_colunas(rows)
reg = []
for r in rows:
    c = None
    for k in r:
        if k.lower().startswith('coment'): c = r[k]; break
    if not c: continue
    reg.append({
        'telegram_topic_id': n(r.get('ID do topico') or r.get('ID do tópico')),
        'id_jogador': s(r.get('ID do jogador')),
        'nome_jogador': r.get('Nome do jogador') or None,
        'comentario': c,
        'data': s(r.get('Data')),
        'telegram_message_id': n(r.get('ID da mensagem') or r.get('telegram_message_id')),
    })
print(f'  Registros validos: {len(reg)}')
upsert('Comentarios_MV', reg)

In [ ]:
print('--- COMENTARIOS VIDEOS')
rows = ler('Comentarios_Videos')
debug_colunas(rows)
reg = []
for r in rows:
    t = r.get('texto') or r.get('Texto')
    if not t:
        for k in r:
            if k.lower().startswith('coment'): t = r[k]; break
    if not t: continue
    reg.append({
        'telegram_topic_id': n(r.get('telegram_topic_id') or r.get('ID do topico')),
        'texto': t,
        'autor': r.get('autor') or r.get('Autor') or None,
        'id_usuario': s(r.get('id_usuario') or r.get('ID do jogador')),
        'data': s(r.get('data') or r.get('Data')),
        'reacoes': s(r.get('reacoes') or r.get('Reacoes')),
        'telegram_message_id': n(r.get('ID da mensagem') or r.get('telegram_message_id')),
    })
print(f'  Registros validos: {len(reg)}')
upsert('Comentarios_Videos', reg)

In [ ]:
print('--- COMENTARIOS ALBUNS')
rows = ler('Comentarios_Albuns')
debug_colunas(rows)
reg = []
for r in rows:
    c = None
    for k in r:
        if k.lower().startswith('coment'): c = r[k]; break
    if not c: continue
    reg.append({
        'telegram_topic_id': n(r.get('ID do topico') or r.get('ID do tópico')),
        'id_jogador': s(r.get('ID do jogador')),
        'nome_jogador': r.get('Nome do jogador') or None,
        'comentario': c,
        'data': s(r.get('Data')),
    })
print(f'  Registros validos: {len(reg)}')
upsert('Comentarios_Albuns', reg)
print('\nIMPORTACAO CONCLUIDA!')